# Computação Gráfica – Projeto 3

### Dupla:

#### Maicon Chaves Marques - 14593530
#### Arthur Trottmann Ramos - 14681052

## Código

### Importação de Bibliotecas

In [253]:
%pip install glfw pyopengl pyglm numpy pillow

import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image
import os
import copy

from Shaders.shader_s import Shader

Note: you may need to restart the kernel to use updated packages.


### Inicialização da Janela

In [254]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)



(python3.13:477297): Gtk-WARNING **: 11:39:20.823: gtk_disable_setlocale() must be called before gtk_init()


### Construção e Linkagem de Shaders

In [255]:
ourShader = Shader("Shaders/vertex_shader.vs", "Shaders/fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Carregamento de Modelos (Vértices e Texturas) via Arquivos

In [256]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_LINE_SMOOTH)


global vertices_list
global textures_coord_list
global normals_list
vertices_list = []
textures_coord_list = []
normals_list = []
    
global number_textures
number_textures = 0


def load_model_from_file(filename):
    '''
    Leitura do .obj para vértices, faces e vértices de texturas
    '''

    vertices = []
    texture_coords = []
    faces = []
    normals = []

    material = None
    object_name = 'default'

    for line in open(filename, "r", encoding="utf-8"):
        if line.startswith('#'): continue
        
        values = line.split()
        if not values: continue

        # Coordenadas dos Vértices do Objeto
        if values[0] == 'v':
            vertices.append(values[1:4])

        # Coordenada da Textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        # Vetores normais
        elif values[0] == 'vn':
            normals.append(values[1:4])

        # Nome do objeto
        elif values[0] == 'o':
            object_name = values[1]

        # Material/Textura
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]

        # Face
        elif values[0] == 'f':

            face = []
            face_texture = []
            face_normal = []

            for v in values[1:]:
                w = v.split('/')

                # Índice do vértice
                face.append(int(w[0]))

                # Índice da textura
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

                # Índice da normal
                if len(w) >= 3 and len(w[2]) > 0:
                    face_normal.append(int(w[2]))
                else:
                    face_normal.append(0)

            faces.append((face, face_texture, face_normal, material, object_name))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces
    model['normals'] = normals

    return model


def load_texture_from_file(texture_id, img_textura):
    '''
    Upload da textura para GPU
    '''

    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]

    if "Material_35_baseColor.png" in img_textura:
        image_data = img.convert("RGBA").tobytes("raw", "RGBA", 0, -1)
        glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, img_width, img_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, image_data)
    
    else:
        image_data = img.convert("RGB").tobytes("raw", "RGB", 0, -1)
        glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)


def circular_sliding_window_of_three(vertices_arr):
    '''
    Converte uma face em triângulos
    '''

    if len(vertices_arr) == 3:
        return vertices_arr
    
    circular_arr = vertices_arr + [vertices_arr[0]]
    result = []

    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])

    return result

def get_sub_object_center(sub):
    verts = vertices_list[sub.start_vertex : sub.start_vertex + sub.quantity_vertex]
    
    xs = [float(v[0]) for v in verts]
    ys = [float(v[1]) for v in verts]
    zs = [float(v[2]) for v in verts]

    # Usando o centro geométrico exato (Bounding Box) em vez da média
    cx = (max(xs) + min(xs)) / 2.0
    cy = (max(ys) + min(ys)) / 2.0
    cz = (max(zs) + min(zs)) / 2.0

    return (cx, cy, cz)

def get_object_center(obj):
    verts = vertices_list[obj.subobjects[0].start_vertex : 
                          obj.subobjects[-1].start_vertex + obj.subobjects[-1].quantity_vertex]
    
    xs = [float(v[0]) for v in verts]
    ys = [float(v[1]) for v in verts]
    zs = [float(v[2]) for v in verts]

    # Mesma correção para o centro global do objeto
    cx = (max(xs) + min(xs)) / 2.0
    cy = (max(ys) + min(ys)) / 2.0
    cz = (max(zs) + min(zs)) / 2.0

    return (cx, cy, cz)

def get_object_bottom_y(obj):
    verts = vertices_list[obj.subobjects[0].start_vertex : 
                          obj.subobjects[-1].start_vertex + obj.subobjects[-1].quantity_vertex]
    
    ys = [float(v[1]) for v in verts]

    return max(ys)

### Classes de Objetos e Subobjetos

In [257]:
class SubObjeto:
    def __init__(self, sub_object_name, material, start_vertex, quantity_vertex):
        self.name = sub_object_name
        self.material_name = material
        self.start_vertex = start_vertex
        self.quantity_vertex = quantity_vertex
    
        self.texture_id = None
        self.color = [1.0, 1.0, 1.0, 1.0]
        self.environment_id = 1

        self.Ns = 1.0
        
        self.Ka = [1.0, 1.0, 1.0]
        self.Kd = [0.0, 0.0, 0.0]
        self.Ks = [1.0, 1.0, 1.0]

        # Centro
        self.pivot_x, self.pivot_y, self.pivot_z = 0, 0, 0

        self.t_x, self.t_y, self.t_z = 0, 0, 0
        self.scale_x, self.scale_y, self.scale_z = 1, 1, 1
        self.angle_x, self.angle_y, self.angle_z = 0, 0, 0

    def set_color(self, r, g, b, a):
        self.color = [r, g, b, a]

    def set_lighting(self, Ns, Ka, Kd, Ks):
        self.Ns = Ns
        self.Ka = Ka
        self.Kd = Kd
        self.Ks = Ks

    def set_transformation_subobject(self, t_x, t_y, t_z, angle_x, angle_y, angle_z, scale_x, scale_y, scale_z):
        self.t_x, self.t_y, self.t_z = t_x, t_y, t_z
        self.angle_x, self.angle_y, self.angle_z = angle_x, angle_y, angle_z
        self.scale_x, self.scale_y, self.scale_z = scale_x, scale_y, scale_z


    def get_world_position(self, parent_object):

        mat_to_origin = model(
            0, 0, 0,
            -parent_object.pivot_x,
            -parent_object.pivot_y,
            -parent_object.pivot_z,
            1, 1, 1
        )

        mat_transform = model(
            parent_object.angle_x,
            parent_object.angle_y,
            parent_object.angle_z,
            parent_object.t_x,
            parent_object.t_y,
            parent_object.t_z,
            parent_object.scale_x,
            parent_object.scale_y,
            parent_object.scale_z
        )

        mat_from_origin = model(
            0, 0, 0,
            parent_object.pivot_x,
            parent_object.pivot_y,
            parent_object.pivot_z,
            1, 1, 1
        )

        mat_global = mat_from_origin @ mat_transform @ mat_to_origin

        sub_to_origin = model(
            0, 0, 0,
            -self.pivot_x,
            -self.pivot_y,
            -self.pivot_z,
            1, 1, 1
        )

        sub_transform = model(
            self.angle_x,
            self.angle_y,
            self.angle_z,
            self.t_x,
            self.t_y,
            self.t_z,
            self.scale_x,
            self.scale_y,
            self.scale_z
        )

        sub_from_origin = model(
            0, 0, 0,
            self.pivot_x,
            self.pivot_y,
            self.pivot_z,
            1, 1, 1
        )

        mat_local = sub_from_origin @ sub_transform @ sub_to_origin

        mat_final = mat_global @ mat_local

        v = vertices_list[self.start_vertex]

        pos = np.array([
            v[0],
            v[1],
            v[2],
            1.0
        ], dtype=np.float32)

        world = mat_final @ pos

        return glm.vec3(
            world[0],
            world[1],
            world[2]
        )
        

class Objeto:
    def __init__(self, obj_file, textures_dict, lighting_dict):
        self.name = obj_file.split('/')[-1].replace('.obj', '')
        self.subobjects = []

        # Centro
        self.pivot_x, self.pivot_y, self.pivot_z = 0, 0, 0

        # Transformações globais do objeto
        self.t_x, self.t_y, self.t_z = 0, 0, 0
        self.scale_x, self.scale_y, self.scale_z = 1, 1, 1
        self.angle_x, self.angle_y, self.angle_z = 0, 0, 0

        self.load(obj_file, textures_dict, lighting_dict)


    def set_transformations(self, t_x, t_y, t_z, angle_x, angle_y, angle_z, scale_x, scale_y, scale_z):
        self.t_x, self.t_y, self.t_z = t_x, t_y, t_z
        self.angle_x, self.angle_y, self.angle_z = angle_x, angle_y, angle_z
        self.scale_x, self.scale_y, self.scale_z = scale_x, scale_y, scale_z

    def scale_object(self, add_x, add_y, add_z):
        self.scale_x += add_x
        self.scale_y += add_y
        self.scale_z += add_z

    def print_info(self):
        print(f"\nObjeto: {self.name}")

        print("Transformações globais:")
        print(f"  Translação: ({self.t_x}, {self.t_y}, {self.t_z})")
        print(f"  Rotação:    ({self.angle_x}, {self.angle_y}, {self.angle_z})")
        print(f"  Escala:     ({self.scale_x}, {self.scale_y}, {self.scale_z})")

    
    def load(self, obj_file, textures_dict, lighting_dict):
        model = load_model_from_file(obj_file)
    
        faces_per_object = {}
    
        for face in model['faces']:
            object_name = face[4]
    
            if object_name not in faces_per_object:
                faces_per_object[object_name] = []
    
            faces_per_object[object_name].append(face)
    
        for object_name, faces in faces_per_object.items():
            start_vertex = len(vertices_list)
    
            material = faces[0][3]
    
            for face in faces:
                vertices = face[0]
                textures = face[1]
                normals = face[2]
    
                for i in range(1, len(vertices) - 1):
                    triangle_vertices = [
                        vertices[0],
                        vertices[i],
                        vertices[i + 1]
                    ]
    
                    triangle_textures = [
                        textures[0],
                        textures[i],
                        textures[i + 1]
                    ]
    
                    triangle_normals = [
                        normals[0],
                        normals[i],
                        normals[i + 1]
                    ]
    
                    for vertex_id in triangle_vertices:
                        vertices_list.append(
                            model['vertices'][vertex_id - 1]
                        )
    
                    for texture_id in triangle_textures:
                        textures_coord_list.append(
                            model['texture'][texture_id - 1]
                        )
    
                    for normal_id in triangle_normals:
                        normals_list.append(
                            model['normals'][normal_id - 1]
                        )
    
            quantidade = len(vertices_list) - start_vertex
    
            sub = SubObjeto(
                object_name,
                material,
                start_vertex,
                quantidade
            )
    
            if material in textures_dict:
                global number_textures
    
                load_texture_from_file(
                    number_textures,
                    textures_dict[material]
                )
    
                sub.texture_id = number_textures
    
                number_textures += 1
    
            if material in lighting_dict:
                light = lighting_dict[material]
    
                sub.Ns = light['Ns']
                sub.Ka = light['Ka']
                sub.Kd = light['Kd']
                sub.Ks = light['Ks']
    
            self.subobjects.append(sub)

    
    def draw(self, program):
        mat_to_origin   = model(0, 0, 0, -self.pivot_x, -self.pivot_y, -self.pivot_z, 1, 1, 1)
        mat_transform   = model(self.angle_x, self.angle_y, self.angle_z, self.t_x, self.t_y, self.t_z, self.scale_x, self.scale_y, self.scale_z)
        mat_from_origin = model(0, 0, 0, self.pivot_x, self.pivot_y, self.pivot_z, 1, 1, 1)
    
        mat_global = mat_from_origin @ mat_transform @ mat_to_origin
    
        loc_model     = glGetUniformLocation(program, "model")
        loc_color     = glGetUniformLocation(program, "color")
        loc_use_color = glGetUniformLocation(program, "use_color")
    
        loc_environment = glGetUniformLocation(program, "environment_id")
    
        for sub in self.subobjects:
    
            glUniform1i(loc_environment, sub.environment_id)
    
            if any([sub.t_x, sub.t_y, sub.t_z,
                    sub.angle_x, sub.angle_y, sub.angle_z,
                    sub.scale_x != 1, sub.scale_y != 1, sub.scale_z != 1]):
                
                sub_to_origin  =  model(0, 0, 0, -sub.pivot_x, -sub.pivot_y, -sub.pivot_z, 1, 1, 1)
                sub_transform  =  model(sub.angle_x, sub.angle_y, sub.angle_z, sub.t_x, sub.t_y, sub.t_z, sub.scale_x, sub.scale_y, sub.scale_z)
                sub_from_origin = model(0, 0, 0, sub.pivot_x, sub.pivot_y, sub.pivot_z, 1, 1, 1)
                
                mat_local = sub_from_origin @ sub_transform @ sub_to_origin
                mat_final = mat_global @ mat_local
            else:
                mat_final = mat_global
    
            glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_final)
    
            if sub.texture_id is not None:
                glBindTexture(GL_TEXTURE_2D, sub.texture_id)
                glUniform1i(loc_use_color, 0)
            else:
                glUniform1i(loc_use_color, 1)
                glUniform4f(loc_color, *sub.color)
            
            loc_Ns = glGetUniformLocation(program, "Ns")
            glUniform1f(loc_Ns, sub.Ns)
            
            loc_Ka = glGetUniformLocation(program, "Ka")
            glUniform3fv(loc_Ka, 1, np.array(sub.Ka, dtype=np.float32))
            
            loc_Kd = glGetUniformLocation(program, "Kd")
            glUniform3fv(loc_Kd, 1, np.array(sub.Kd, dtype=np.float32))
            
            loc_Ks = glGetUniformLocation(program, "Ks")
            glUniform3fv(loc_Ks, 1, np.array(sub.Ks, dtype=np.float32))
            
            glDrawArrays(GL_TRIANGLES, sub.start_vertex, sub.quantity_vertex)

### Carregamento dos Objetos

In [258]:
Objects = []

# =========================
# DEFINIÇÃO DOS OBJETOS
# =========================

Barril = Objeto(
    obj_file='Objetos/Barril/barril.obj',
    textures_dict={
        'Oak_Wood_Floor': 'Objetos/Barril/Oak_Wood_Floor_baseColor.jpeg',
        'Rust_3.001': 'Objetos/Barril/Rust_3.001_baseColor.jpeg'
    },
    lighting_dict={
        'Oak_Wood_Floor': {'Ns': 90.0, 'Ka': [0.34, 0.24, 0.16], 'Kd': [0.92, 0.62, 0.34], 'Ks': [0.18, 0.14, 0.10]},
        'Rust_3.001': {'Ns': 18.0, 'Ka': [0.22, 0.16, 0.12], 'Kd': [0.58, 0.30, 0.16], 'Ks': [0.03, 0.03, 0.03]}
    }
)

Cacto01 = Objeto(
    obj_file='Objetos/Cacto01/cacto01.obj',
    textures_dict={
        'cactus_0Mat': 'Objetos/Cacto01/cactus_0Mat_baseColor.png',
        'cactus_blend_1Mat': 'Objetos/Cacto01/cactus_blend_1Mat_baseColor.png',
        'prickly_pear_4Mat': 'Objetos/Cacto01/prickly_pear_4Mat_baseColor.png',
        'prickly_pear_blend_3Mat': 'Objetos/Cacto01/prickly_pear_blend_3Mat_baseColor.png',
        'prickly_pear_flower_9Mat': 'Objetos/Cacto01/prickly_pear_flower_9Mat_baseColor.png',
        'prickly_pear_stem_5Mat': 'Objetos/Cacto01/prickly_pear_stem_5Mat_baseColor.png',
        'saguaro_flower_7Mat': 'Objetos/Cacto01/saguaro_flower_7Mat_baseColor.png',
        'saguaro_stem_2Mat': 'Objetos/Cacto01/saguaro_stem_2Mat_baseColor.png',
        'spines_01_6Mat': 'Objetos/Cacto01/spines_01_6Mat_baseColor.png',
        'spines_02_8Mat': 'Objetos/Cacto01/spines_02_8Mat_baseColor.png',
    },
    lighting_dict={
        'cactus_0Mat': {'Ns': 6.0, 'Ka': [0.22, 0.32, 0.18], 'Kd': [0.28, 0.95, 0.18], 'Ks': [0.0, 0.0, 0.0]},
        'cactus_blend_1Mat': {'Ns': 6.0, 'Ka': [0.24, 0.34, 0.20], 'Kd': [0.42, 0.98, 0.26], 'Ks': [0.0, 0.0, 0.0]},
        'prickly_pear_4Mat': {'Ns': 5.0, 'Ka': [0.22, 0.32, 0.18], 'Kd': [0.34, 0.92, 0.20], 'Ks': [0.0, 0.0, 0.0]},
        'prickly_pear_blend_3Mat': {'Ns': 5.0, 'Ka': [0.24, 0.34, 0.20], 'Kd': [0.45, 1.0, 0.28], 'Ks': [0.0, 0.0, 0.0]},
        'prickly_pear_flower_9Mat': {'Ns': 28.0, 'Ka': [0.48, 0.34, 0.12], 'Kd': [1.0, 0.82, 0.18], 'Ks': [0.12, 0.10, 0.04]},
        'prickly_pear_stem_5Mat': {'Ns': 5.0, 'Ka': [0.20, 0.30, 0.16], 'Kd': [0.18, 0.75, 0.10], 'Ks': [0.0, 0.0, 0.0]},
        'saguaro_flower_7Mat': {'Ns': 40.0, 'Ka': [0.44, 0.44, 0.34], 'Kd': [1.0, 1.0, 0.82], 'Ks': [0.18, 0.18, 0.16]},
        'saguaro_stem_2Mat': {'Ns': 5.0, 'Ka': [0.20, 0.30, 0.16], 'Kd': [0.20, 0.82, 0.12], 'Ks': [0.0, 0.0, 0.0]},
        'spines_01_6Mat': {'Ns': 120.0, 'Ka': [0.30, 0.24, 0.18], 'Kd': [0.88, 0.78, 0.62], 'Ks': [0.55, 0.48, 0.40]},
        'spines_02_8Mat': {'Ns': 120.0, 'Ka': [0.28, 0.22, 0.16], 'Kd': [0.82, 0.70, 0.52], 'Ks': [0.55, 0.48, 0.40]}
    }
)

Cacto02 = Objeto(
    obj_file='Objetos/Cacto02/cacto02.obj',
    textures_dict={
        'agave_2Mat': 'Objetos/Cacto02/agave_2Mat_baseColor.png',
        'bark_02_0Mat': 'Objetos/Cacto02/bark_02_0Mat_baseColor.png',
        'bark_1Mat': 'Objetos/Cacto02/bark_1Mat_baseColor.png',
        'cholla_01_9Mat': 'Objetos/Cacto02/cholla_01_9Mat_baseColor.png',
        'cholla_02_8Mat': 'Objetos/Cacto02/cholla_02_8Mat_baseColor.png',
        'creosote_branch_02_7Mat': 'Objetos/Cacto02/creosote_branch_02_7Mat_baseColor.png',
        'ocotillo_branch_01_5Mat': 'Objetos/Cacto02/ocotillo_branch_01_5Mat_baseColor.png',
        'ocotillo_branch_02_6Mat': 'Objetos/Cacto02/ocotillo_branch_02_6Mat_baseColor.png',
        'yacca_leaf_01_3Mat': 'Objetos/Cacto02/yacca_leaf_01_3Mat_baseColor.png',
        'yacca_leaf_02_4Mat': 'Objetos/Cacto02/yacca_leaf_02_4Mat_baseColor.png',
    },
    lighting_dict={
        'agave_2Mat': {'Ns': 5.0, 'Ka': [0.22, 0.32, 0.18], 'Kd': [0.30, 0.88, 0.16], 'Ks': [0.0, 0.0, 0.0]},
        'bark_02_0Mat': {'Ns': 24.0, 'Ka': [0.28, 0.22, 0.18], 'Kd': [0.78, 0.62, 0.46], 'Ks': [0.05, 0.05, 0.05]},
        'bark_1Mat': {'Ns': 20.0, 'Ka': [0.22, 0.16, 0.10], 'Kd': [0.42, 0.22, 0.10], 'Ks': [0.03, 0.03, 0.03]},
        'cholla_01_9Mat': {'Ns': 10.0, 'Ka': [0.24, 0.20, 0.16], 'Kd': [0.62, 0.52, 0.42], 'Ks': [0.02, 0.02, 0.02]},
        'cholla_02_8Mat': {'Ns': 18.0, 'Ka': [0.20, 0.16, 0.10], 'Kd': [0.48, 0.28, 0.12], 'Ks': [0.03, 0.03, 0.03]},
        'creosote_branch_02_7Mat': {'Ns': 5.0, 'Ka': [0.20, 0.30, 0.16], 'Kd': [0.22, 0.72, 0.14], 'Ks': [0.0, 0.0, 0.0]},
        'ocotillo_branch_01_5Mat': {'Ns': 5.0, 'Ka': [0.20, 0.30, 0.16], 'Kd': [0.20, 0.68, 0.12], 'Ks': [0.0, 0.0, 0.0]},
        'ocotillo_branch_02_6Mat': {'Ns': 5.0, 'Ka': [0.20, 0.30, 0.16], 'Kd': [0.22, 0.64, 0.10], 'Ks': [0.0, 0.0, 0.0]},
        'yacca_leaf_01_3Mat': {'Ns': 4.0, 'Ka': [0.20, 0.32, 0.16], 'Kd': [0.18, 0.78, 0.08], 'Ks': [0.0, 0.0, 0.0]},
        'yacca_leaf_02_4Mat': {'Ns': 8.0, 'Ka': [0.22, 0.20, 0.16], 'Kd': [0.52, 0.40, 0.24], 'Ks': [0.01, 0.01, 0.01]}
    }
)

Cadeira = Objeto(
    obj_file='Objetos/Cadeira/cadeira.obj',
    textures_dict={
        'Material.002': 'Objetos/Cadeira/Chair_albedo.png'
    },
    lighting_dict={
        'Material.002': {'Ns': 110.0, 'Ka': [0.30, 0.18, 0.10], 'Kd': [0.88, 0.38, 0.16], 'Ks': [0.28, 0.22, 0.18]}
    }
)


Cama = Objeto(
    obj_file='Objetos/Cama/cama.obj',
    textures_dict={
        'Material.001': 'Objetos/Cama/internals_albedo.png'
    },
    lighting_dict={
        'Material.001': {'Ns': 4.0, 'Ka': [0.24, 0.24, 0.24], 'Kd': [0.42, 0.42, 0.42], 'Ks': [0.0, 0.0, 0.0]}
    }
)

Casa = Objeto(
    obj_file='Objetos/Casa/casa.obj',
    textures_dict={
        'Ablakok': 'Objetos/Casa/Ablakok_baseColor.jpeg',
        'AblakosKulso': 'Objetos/Casa/AblakosKulso_baseColor.jpeg',
        'Ajto': 'Objetos/Casa/FentiAjto_baseColor.jpeg',
        'Bejarat': 'Objetos/Casa/Bejarat_baseColor.jpeg',
        'EmeletTalaj': 'Objetos/Casa/EmeletTalaj_baseColor.jpeg',
        'FentiAjto': 'Objetos/Casa/FentiAjto_baseColor.jpeg',
        'Korlat': 'Objetos/Casa/Korlat_baseColor.jpeg',
        'KulsoEsOszlopok': 'Objetos/Casa/KulsoEsOszlopokFent_baseColor.jpeg',
        'KulsoEsOszlopokFent': 'Objetos/Casa/KulsoEsOszlopokFent_baseColor.jpeg',
        'KulsoFal': 'Objetos/Casa/KulsoFal_baseColor.jpeg',
        'Lepcso': 'Objetos/Casa/Lepcso_baseColor.jpeg',
        'Polc': 'Objetos/Casa/Polc_baseColor.jpeg',
        'Pult': 'Objetos/Casa/Pult_baseColor.jpeg',
        'Talaj': 'Objetos/Casa/Talaj_baseColor.jpeg',
        'Teto': 'Objetos/Casa/Teto_baseColor.jpeg',
    },
    lighting_dict={
        'Ablakok': {'Ns': 180.0, 'Ka': [0.18, 0.12, 0.08], 'Kd': [0.42, 0.18, 0.08], 'Ks': [0.85, 0.78, 0.72]},
        'AblakosKulso': {'Ns': 65.0, 'Ka': [0.28, 0.22, 0.16], 'Kd': [0.88, 0.68, 0.44], 'Ks': [0.16, 0.14, 0.12]},
        'Ajto': {'Ns': 85.0, 'Ka': [0.24, 0.16, 0.10], 'Kd': [0.62, 0.24, 0.10], 'Ks': [0.20, 0.16, 0.12]},
        'Bejarat': {'Ns': 70.0, 'Ka': [0.30, 0.24, 0.18], 'Kd': [0.92, 0.70, 0.46], 'Ks': [0.18, 0.16, 0.14]},
        'EmeletTalaj': {'Ns': 40.0, 'Ka': [0.26, 0.18, 0.10], 'Kd': [0.75, 0.42, 0.16], 'Ks': [0.08, 0.06, 0.05]},
        'FentiAjto': {'Ns': 85.0, 'Ka': [0.24, 0.16, 0.10], 'Kd': [0.62, 0.24, 0.10], 'Ks': [0.20, 0.16, 0.12]},
        'Korlat': {'Ns': 140.0, 'Ka': [0.18, 0.12, 0.08], 'Kd': [0.38, 0.14, 0.06], 'Ks': [0.45, 0.36, 0.28]},
        'KulsoEsOszlopok': {'Ns': 45.0, 'Ka': [0.22, 0.18, 0.12], 'Kd': [0.52, 0.42, 0.24], 'Ks': [0.08, 0.08, 0.06]},
        'KulsoEsOszlopokFent': {'Ns': 45.0, 'Ka': [0.22, 0.18, 0.12], 'Kd': [0.52, 0.42, 0.24], 'Ks': [0.08, 0.08, 0.06]},
        'KulsoFal': {'Ns': 8.0, 'Ka': [0.34, 0.28, 0.22], 'Kd': [0.95, 0.70, 0.42], 'Ks': [0.01, 0.01, 0.01]},
        'Lepcso': {'Ns': 100.0, 'Ka': [0.18, 0.12, 0.08], 'Kd': [0.36, 0.12, 0.05], 'Ks': [0.26, 0.20, 0.16]},
        'Polc': {'Ns': 70.0, 'Ka': [0.22, 0.16, 0.10], 'Kd': [0.42, 0.22, 0.14], 'Ks': [0.14, 0.12, 0.10]},
        'Pult': {'Ns': 120.0, 'Ka': [0.18, 0.12, 0.08], 'Kd': [0.34, 0.14, 0.04], 'Ks': [0.32, 0.26, 0.20]},
        'Talaj': {'Ns': 18.0, 'Ka': [0.24, 0.18, 0.12], 'Kd': [0.58, 0.30, 0.12], 'Ks': [0.02, 0.02, 0.02]},
        'Teto': {'Ns': 30.0, 'Ka': [0.26, 0.18, 0.12], 'Kd': [0.72, 0.40, 0.22], 'Ks': [0.05, 0.05, 0.04]}
    }
)

Cavalo = Objeto(
    obj_file='Objetos/Cavalo/cavalo.obj',
    textures_dict={
        'Body': 'Objetos/Cavalo/Body_albedo2.jpg',
        'Cannon_A.001': 'Objetos/Cavalo/Cannon_A_AlbedoTransparency.png',
        'Cannon_A_Metal.001': 'Objetos/Cavalo/Cannon_A_Metal_AlbedoTransparency.png',
        'Cannon_A_Wood.001': 'Objetos/Cavalo/Cannon_A_Wood_AlbedoTransparency.png',
        'Eyes': 'Objetos/Cavalo/Eye_brown_albedo.jpg',
        'Hair': 'Objetos/Cavalo/Neck_albedo.jpg',
        'Voorwagen_Harnass': 'Objetos/Cavalo/Voorwagen_Harnass_AlbedoTransparency.png',
        'Voorwagen_Metal': 'Objetos/Cavalo/Voorwagen_Metal_AlbedoTransparency.png',
        'Voorwagen_Wood': 'Objetos/Cavalo/Voorwagen_Wood_AlbedoTransparency.png',
    },
    lighting_dict={
        'Body': {'Ns': 24.0, 'Ka': [0.28, 0.20, 0.14], 'Kd': [0.82, 0.52, 0.30], 'Ks': [0.04, 0.04, 0.03]},
        'Cannon_A.001': {'Ns': 180.0, 'Ka': [0.20, 0.20, 0.18], 'Kd': [0.52, 0.48, 0.34], 'Ks': [0.82, 0.82, 0.76]},
        'Cannon_A_Metal.001': {'Ns': 1000.0, 'Ka': [0.18, 0.18, 0.18], 'Kd': [0.32, 0.32, 0.32], 'Ks': [1.0, 1.0, 1.0]},
        'Cannon_A_Wood.001': {'Ns': 90.0, 'Ka': [0.22, 0.16, 0.10], 'Kd': [0.52, 0.34, 0.22], 'Ks': [0.18, 0.14, 0.10]},
        'Eyes': {'Ns': 1000.0, 'Ka': [0.18, 0.14, 0.10], 'Kd': [0.38, 0.22, 0.12], 'Ks': [1.0, 1.0, 1.0]},
        'Hair': {'Ns': 10.0, 'Ka': [0.24, 0.18, 0.10], 'Kd': [0.72, 0.46, 0.14], 'Ks': [0.01, 0.01, 0.01]},
        'Voorwagen_Harnass': {'Ns': 60.0, 'Ka': [0.20, 0.14, 0.10], 'Kd': [0.42, 0.20, 0.10], 'Ks': [0.12, 0.10, 0.08]},
        'Voorwagen_Metal': {'Ns': 1000.0, 'Ka': [0.16, 0.16, 0.16], 'Kd': [0.28, 0.28, 0.28], 'Ks': [1.0, 1.0, 1.0]},
        'Voorwagen_Wood': {'Ns': 80.0, 'Ka': [0.22, 0.16, 0.10], 'Kd': [0.58, 0.38, 0.22], 'Ks': [0.18, 0.14, 0.10]}
    }
)

Ceu = Objeto(
    obj_file='Objetos/Ceu/ceu.obj',
    textures_dict={
        'Material.001': 'Objetos/Ceu/space-skybox-texture-mapping-cube-mapping-night-sky-24df7747449631f3f2a45fc630ae6ad0.png'
    },
    lighting_dict={
        'Material.001': {'Ns': 1.0, 'Ka': [0.08, 0.10, 0.16], 'Kd': [0.06, 0.10, 0.18], 'Ks': [0.0, 0.0, 0.0]}
    }
)

Chao = Objeto(
    obj_file='Objetos/Chao/chao.obj',
    textures_dict={
        'Materiais': 'Objetos/Chao/4_Albedo.png'
    },
    lighting_dict={
        'Materiais': {'Ns': 2.0, 'Ka': [0.34, 0.24, 0.16], 'Kd': [1.0, 0.72, 0.42], 'Ks': [0.0, 0.0, 0.0]}
    }
)

Jhon = Objeto(
    obj_file='Objetos/JMeTocha/jmtocha.obj',
    textures_dict={
        '.bmp': 'Objetos/JMeTocha/repeater_carbine01.png',
        'iv_drawable_material.bmp.001': 'Objetos/JMeTocha/melee_lasso01.png',
        'player_default_intro_alphakill_d1.bmp': 'Objetos/JMeTocha/player_default_intro_alphakill_d1.png',
        'torchstick01': 'Objetos/JMeTocha/torchstick_baseColor.png',
        'torchstick02': 'Objetos/JMeTocha/torchstick_baseColor.png'
    },
    lighting_dict={
        '.bmp': {'Ns': 180.0, 'Ka': [0.60, 0.60, 0.60], 'Kd': [0.65, 0.65, 0.65], 'Ks': [1.0, 1.0, 1.0]},
        'iv_drawable_material.bmp.001': {'Ns': 12.0, 'Ka': [0.58, 0.48, 0.34], 'Kd': [1.00, 0.85, 0.55], 'Ks': [0.20, 0.16, 0.10]},
        'player_default_intro_alphakill_d1.bmp': {'Ns': 8.0, 'Ka': [0.55, 0.48, 0.40], 'Kd': [0.85, 0.70, 0.55], 'Ks': [0.10, 0.10, 0.10]},
        'torchstick01': {'Ns': 20.0, 'Ka': [0.60, 0.45, 0.28], 'Kd': [1.00, 0.80, 0.35], 'Ks': [0.25, 0.20, 0.12]},
        'torchstick02': {'Ns': 0, 'Ka': [1.0, 1.0, 1.0], 'Kd': [1.0, 1.0, 1.0], 'Ks': [1.0, 1.0, 1.0]}
    }
)

Vela = Objeto(
    obj_file='Objetos/Vela/vela.obj',
    textures_dict={
        'Material_31': 'Objetos/Vela/Material_31_baseColor.jpeg',
        'Material_33': 'Objetos/Vela/Material_33_baseColor.jpeg',
        'Material_35': 'Objetos/Vela/Material_35_baseColor.png',
    },
    lighting_dict={
        'Material_31': {'Ns': 45.0, 'Ka': [0.34, 0.24, 0.12], 'Kd': [1.0, 0.68, 0.16], 'Ks': [0.12, 0.10, 0.06]},
        'Material_33': {'Ns': 10.0, 'Ka': [0.26, 0.20, 0.14], 'Kd': [0.78, 0.58, 0.34], 'Ks': [0.01, 0.01, 0.01]},
        'Material_35': {'Ns': 0, 'Ka': [1.0, 1.0, 1.0], 'Kd': [0.0, 0.0, 0.0], 'Ks': [0.0, 0.0, 0.0]}
    }
)

Lampiao = Objeto(
    obj_file='Objetos/Lampiao/lampiao.obj',
    textures_dict={
        'Glass_2.3': 'Objetos/Lampiao/Glass_2.3_baseColor.png',
        'Lamp_body': 'Objetos/Lampiao/Lamp_body_baseColor.png',
    },
    lighting_dict={
        'Flame': {'Ns': 1000.0, 'Ka': [1.0, 0.72, 0.18], 'Kd': [1.0, 0.75, 0.12], 'Ks': [1.0, 0.92, 0.55]},
        'Glass_2.3': {'Ns': 5000.0, 'Ka': [1.0, 1.0, 1.0], 'Kd': [1.0, 1.0, 1.0], 'Ks': [1.0, 1.0, 1.0]},
        'Lamp_body': {'Ns': 260.0, 'Ka': [0.22, 0.18, 0.14], 'Kd': [0.52, 0.46, 0.40], 'Ks': [0.82, 0.78, 0.72]}
    }
)

Mapa = Objeto(
    obj_file='Objetos/Mapa/mapa.obj',
    textures_dict={
        'TreasureMap': 'Objetos/Mapa/TreasureMap_baseColor.png'
    },
    lighting_dict={
        'TreasureMap': {'Ns': 3.0, 'Ka': [0.32, 0.26, 0.18], 'Kd': [0.92, 0.82, 0.52], 'Ks': [0.0, 0.0, 0.0]}
    }
)

Mesa = Objeto(
    obj_file='Objetos/Mesa/mesa.obj',
    textures_dict={
        'Material.002': 'Objetos/Mesa/Asztal_albedo.png'
    },
    lighting_dict={
        'Material.002': {'Ns': 0, 'Ka': [0.30, 0.18, 0.10], 'Kd': [0.95, 0.42, 0.16], 'Ks': [0.34, 0.26, 0.18]}
    }
)

Rifle = Objeto(
    obj_file='Objetos/Rifle/rifle.obj',
    textures_dict={
        'WoodenSmall': 'Objetos/Rifle/WoodenSmall_albedo.jpg',
        'WoodenBig': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'Pipe': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'Cock': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'MetalBody': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'MetalPart': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'PipeBetween': 'Objetos/Rifle/WoodenBig_albedo.jpg'
    },
    lighting_dict={
        'Cock': {'Ns': 1000.0, 'Ka': [0.18, 0.18, 0.18], 'Kd': [0.38, 0.42, 0.46], 'Ks': [1.0, 1.0, 1.0]},
        'MetalBody': {'Ns': 1000.0, 'Ka': [0.18, 0.18, 0.18], 'Kd': [0.38, 0.42, 0.46], 'Ks': [1.0, 1.0, 1.0]},
        'MetalPart': {'Ns': 1000.0, 'Ka': [0.18, 0.18, 0.18], 'Kd': [0.38, 0.42, 0.46], 'Ks': [1.0, 1.0, 1.0]},
        'Pipe': {'Ns': 1000.0, 'Ka': [0.18, 0.18, 0.18], 'Kd': [0.38, 0.42, 0.46], 'Ks': [1.0, 1.0, 1.0]},
        'PipeBetween': {'Ns': 1000.0, 'Ka': [0.18, 0.18, 0.18], 'Kd': [0.38, 0.42, 0.46], 'Ks': [1.0, 1.0, 1.0]},
        'WoodenBig': {'Ns': 120.0, 'Ka': [0.28, 0.16, 0.10], 'Kd': [0.78, 0.24, 0.12], 'Ks': [0.24, 0.18, 0.12]},
        'WoodenSmall': {'Ns': 90.0, 'Ka': [0.24, 0.14, 0.08], 'Kd': [0.58, 0.16, 0.08], 'Ks': [0.18, 0.12, 0.08]}
    }
)

# =========================
# TRANSFORMAÇÕES
# =========================

Barril.set_transformations(-20.5, 0, -16, 0.0, 0.0, 0.0, 2.30, 2.30, 2.30)
Cacto01.set_transformations(-56, 0.5, 4.5, 0.0, 0.0, 0.0, 2.20, 2.20, 2.20)
Cacto02.set_transformations(-38, 0.2, 36, 0.0, 0.0, 0.0, 3.51, 3.51, 3.51)
Cadeira.set_transformations(-24.5, -1.5, -1.5, 0.0, -90.0, 0.0, 1.40, 1.40, 1.40)
Cama.set_transformations(-32.38, 1, 8.5, 0, 0, 0, 0.3, 0.3, 0.3)
Casa.set_transformations(-28, 15, 0, 0, 0, 0, 3, 3, 3)
Cavalo.set_transformations(-49.625000, -1.750000, -16.500000, 0.0, 135.0, 0.0, 2.80, 2.80, 2.80)
Ceu.set_transformations(0, 0, 0, 0, 0, 0, 2, 2, 2)
Chao.set_transformations(0, 0, 0, 0, 0, 0, 40, 40, 40)
Jhon.set_transformations(-53.02, 0.0, -10.48, 0.0, -140.0, 0.0, 7.60, 7.60, 7.60)
Vela.set_transformations(-23.41, 4.33, 7.42,0.0, -140.0, 0.0, 0.79, 0.79, 0.79)
Lampiao.set_transformations(-23.5, 2.1875, -1.5, 0, 0, 0, 1, 1, 1)
Mapa.set_transformations(-20.75, 3.28125, 6.625, 32.5, -180.0, 0.0, 0.10, 0.10, 0.10)
Mesa.set_transformations(-23.5, -5, -1.5, 0, 0, 0, 2.5, 2.5, 2.5)
Rifle.set_transformations(-26.140625, 4.375000, 14.843750, -5.0, 20.0, -15.0, 1.40, 1.40, 1.40)

# =========================
# APPEND & Set dos Centros dos Objetos
# =========================

Objects.extend([
    Barril, Cacto01, Cacto02, Cadeira, Cama, Casa,
    Cavalo, Ceu, Chao, Jhon, Vela, Lampiao,
    Mapa, Mesa, Rifle
])


Cavalo.pivot_x, Cavalo.pivot_y, Cavalo.pivot_z = get_object_center(Cavalo)
Rifle.pivot_x, Rifle.pivot_y, Rifle.pivot_z = get_object_center(Rifle)

wheels_index = [3, 4, 12, 13]

for i in wheels_index:
    sub = Cavalo.subobjects[i]
    cx, cy, cz = get_sub_object_center(sub)
    sub.pivot_x, sub.pivot_y, sub.pivot_z = cx, cy, cz

for sub in Barril.subobjects:
    sub.environment_id = 2

for sub in Cacto01.subobjects:
    sub.environment_id = 2

for sub in Cacto02.subobjects:
    sub.environment_id = 2

for sub in Cadeira.subobjects:
    sub.environment_id = 1

for sub in Cama.subobjects:
    sub.environment_id = 1

for sub in Casa.subobjects:
    sub.environment_id = 0

for sub in Cavalo.subobjects:
    sub.environment_id = 2

for sub in Ceu.subobjects:
    sub.environment_id = 2

for sub in Chao.subobjects:
    sub.environment_id = 2

for sub in Jhon.subobjects:
    sub.environment_id = 2

for sub in Vela.subobjects:
    sub.environment_id = 1

for sub in Lampiao.subobjects:
    sub.environment_id = 1

for sub in Mapa.subobjects:
    sub.environment_id = 1

for sub in Mesa.subobjects:
    sub.environment_id = 1

for sub in Rifle.subobjects:
    sub.environment_id = 1

light_colors = [[1, 1, 1], [1, 0, 0], [1, 1, 1]]
light_types_enabled = [True, True, True]

Objects2 = copy.deepcopy(Objects)

### Enviando os dados da CPU para a GPU, requisitando dois slots (buffers): um para os vértices e outro para as texturas.

In [259]:
buffer_VBO = glGenBuffers(3)

### Enviando coordenadas de vértices para a GPU

In [260]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [261]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Enviando dados de iluminação: vetores normais para a GPU

In [262]:
normals = np.zeros(len(normals_list), [("position", np.float32, 3)])
normals['position'] = normals_list

glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)

stride = normals.strides[0]
offset = ctypes.c_void_p(0)

loc_normals = glGetAttribLocation(program, "normal")

glEnableVertexAttribArray(loc_normals)

glVertexAttribPointer(loc_normals, 3, GL_FLOAT, False, stride, offset)

### Eventos para modificar a posição da câmera.

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [263]:
def alterar_material(atributo, incremento):

    for obj in Objects:
        for sub in obj.subobjects:

            # Ns
            if atributo == "Ns":

                sub.Ns = max(sub.Ns + incremento, 0.0)

            # Ka, Kd, Ks
            else:

                valor_atual = getattr(sub, atributo)

                # trava no limite superior
                #if incremento > 0 and max(valor_atual) >= 1.0:
                #    continue

                # trava no limite inferior
                #if incremento < 0 and min(valor_atual) <= 0.0:
                #    continue

                novo_valor = [
                    k + incremento
                    for k in valor_atual
                ]

                setattr(sub, atributo, novo_valor)

# Camera configs
cameraPos   = glm.vec3(-52, 15, -69.0)
cameraFront = glm.vec3(0.0, 0.0, 1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   =  90.0
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

X_MIN = -201
X_MAX = 201
Y_MIN = 1
Y_MAX = 201
Z_MIN = -201
Z_MAX = 201

deltaTime = 0.0
lastFrame = 0.0


def key_event(window, key, scancode, action, mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode, light_types_enabled
    global Objects, Barril, Cacto01, Cacto02, Cadeira, Cama, Casa, Cavalo, Ceu, Chao, Jhon, Vela, Lampiao, Mapa, Mesa, Rifle

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)

    # --- Camera: WASD ---
    cameraSpeed = 250 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    cameraPos.x = max(X_MIN, min(X_MAX, cameraPos.x))
    cameraPos.y = max(Y_MIN, min(Y_MAX, cameraPos.y))
    cameraPos.z = max(Z_MIN, min(Z_MAX, cameraPos.z))

    if key == glfw.KEY_UP: Jhon.t_x += 0.3
    if key == glfw.KEY_DOWN: Jhon.t_x -= 0.3
    if key == glfw.KEY_RIGHT: Jhon.t_z += 0.3
    if key == glfw.KEY_LEFT: Jhon.t_z -= 0.3

    if action == glfw.PRESS:

        if key == glfw.KEY_X:
            Objects = copy.deepcopy(Objects2)
    
            (
                Barril, Cacto01, Cacto02, Cadeira, Cama, Casa,
                Cavalo, Ceu, Chao, Jhon, Vela, Lampiao,
                Mapa, Mesa, Rifle
            ) = Objects
    
            light_types_enabled = [True, True, True]
    
        if key == glfw.KEY_E:
    
            if Jhon.subobjects[8].environment_id == 2:
                Jhon.subobjects[8].environment_id = -1
                Jhon.subobjects[8].Ns = 5000.0
            else:
                Jhon.subobjects[8].environment_id = 2
                Jhon.subobjects[8].Ns = 0
    
    
        elif key == glfw.KEY_R:
    
            if Vela.subobjects[5].environment_id == 1:
                Vela.subobjects[5].environment_id = -2
                Vela.subobjects[4].Ns = 5000.0
                Vela.subobjects[5].Ns = 5000.0
                Vela.subobjects[4].Ka = [0.3, 0.3, 0.3]
                Vela.subobjects[5].Ka = [0.3, 0.3, 0.3]
            else:
                Vela.subobjects[5].environment_id = 1
                Vela.subobjects[4].Ns = 0
                Vela.subobjects[5].Ns = 0
                Vela.subobjects[4].Ka = [1.0, 1.0, 1.0]
                Vela.subobjects[5].Ka = [1.0, 1.0, 1.0]
    
    
        elif key == glfw.KEY_T:
    
            if Lampiao.subobjects[1].environment_id == 1:
                Lampiao.subobjects[1].environment_id = -3
                Lampiao.subobjects[1].Ns = 5000.0
                Lampiao.subobjects[1].Ka = [0.3, 0.3, 0.3]
            else:
                Lampiao.subobjects[1].environment_id = 1
                Lampiao.subobjects[1].Ns = 0
                Lampiao.subobjects[1].Ka = [1.0, 1.0, 1.0]
    
        elif key == glfw.KEY_V:
            light_types_enabled[0] = not light_types_enabled[0]
    
    
        elif key == glfw.KEY_B:
            light_types_enabled[1] = not light_types_enabled[1]
    
    
        elif key == glfw.KEY_N:
            light_types_enabled[2] = not light_types_enabled[2]

    if action == glfw.PRESS or action == glfw.REPEAT:
    
        if key == glfw.KEY_Y:
            alterar_material("Ns", 3.0)
    
        elif key == glfw.KEY_U:
            alterar_material("Ka", 0.05)
    
        elif key == glfw.KEY_I:
            alterar_material("Kd", 0.05)
    
        elif key == glfw.KEY_O:
            alterar_material("Ks", 0.05)
    
        elif key == glfw.KEY_G:
            alterar_material("Ns", -3.0)
    
        elif key == glfw.KEY_H:
            alterar_material("Ka", -0.05)
    
        elif key == glfw.KEY_J:
            alterar_material("Kd", -0.05)
    
        elif key == glfw.KEY_K:
            alterar_material("Ks", -0.05)

def framebuffer_size_callback(window, largura, altura):
    glViewport(0, 0, largura, altura)


def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch

    if firstMouse:
        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = (xpos - lastX) * 0.1
    yoffset = (lastY - ypos) * 0.1
    lastX = xpos
    lastY = ypos

    yaw   += xoffset
    pitch += yoffset
    pitch  = max(-89.0, min(89.0, pitch))

    front = glm.vec3(
        glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch)),
        glm.sin(glm.radians(pitch)),
        glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    )
    cameraFront = glm.normalize(front)


def scroll_callback(window, xoffset, yoffset):
    global fov
    fov = max(1.0, min(45.0, fov - yoffset))


glfw.set_key_callback(window, key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [264]:
def model(angle_x, angle_y, angle_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat = glm.mat4(1.0)

    mat = glm.translate(mat, glm.vec3(t_x, t_y, t_z))

    mat = glm.rotate(mat, glm.radians(angle_x), glm.vec3(1, 0, 0))
    mat = glm.rotate(mat, glm.radians(angle_y), glm.vec3(0, 1, 0))
    mat = glm.rotate(mat, glm.radians(angle_z), glm.vec3(0, 0, 1))

    mat = glm.scale(mat, glm.vec3(s_x, s_y, s_z))

    return np.array(mat)

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0)

    
    mat_projection = np.array(mat_projection)
    return mat_projection

### Exibindo a janela!


In [265]:
glfw.show_window(window)

### Loop principal da janela.

In [266]:
glEnable(GL_DEPTH_TEST)

polygonal_mode = False

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events()

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glClearColor(1.0, 1.0, 1.0, 1.0)

    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    light_environments = [
        Jhon.subobjects[8].environment_id,
        Vela.subobjects[5].environment_id,
        Lampiao.subobjects[1].environment_id
    ]

    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    lightPositions = [
        Jhon.subobjects[8].get_world_position(Jhon),
        Vela.subobjects[5].get_world_position(Vela),
        Lampiao.subobjects[1].get_world_position(Lampiao)
    ]

    for i, lightPos in enumerate(lightPositions):

        glUniform3f(
            glGetUniformLocation(program, f"lightPos[{i}]"),
            lightPos.x,
            lightPos.y,
            lightPos.z
        )

        glUniform3f(
            glGetUniformLocation(program, f"lightColor[{i}]"),
            light_colors[i][0],
            light_colors[i][1],
            light_colors[i][2]
        )

        glUniform1i(
            glGetUniformLocation(program, f"lightEnvironment[{i}]"),
            light_environments[i]
        )

    for i, enabled in enumerate(light_types_enabled):

        glUniform1i(
            glGetUniformLocation(program, f"enabledLightTypes[{i}]"),
            int(enabled)
        )

    loc_viewPos = glGetUniformLocation(program, "viewPos")

    glUniform3f(
        loc_viewPos,
        cameraPos.x,
        cameraPos.y,
        cameraPos.z
    )

    for obj in Objects:
        obj.draw(program)

    glfw.swap_buffers(window)

glfw.terminate()